# 🔥 烽火南境小说生成器 - 一键构建 APK

## 使用步骤：
1. 点击菜单 **运行时 → 全部运行**
2. 第2步会弹出文件上传框，选择 `novel_generator.zip`
3. 等待构建完成（首次约15-30分钟）
4. 第6步自动下载 APK

## zip 文件制作：
在项目目录 `小说生成器` 下，将以下文件打包成 `novel_generator.zip`：
```
novel_generator.zip
├── main.py
├── novel_core.py
├── 动态AI痕迹消除系统.py
├── ultimate_ai_humanizer.py
├── 章节生成提示词辅助模块.py
├── 顶流作者人格建模辅助模块.py
├── 工具人角色检测系统.py
├── buildozer.spec
├── books/
├── chapters/
├── database/
└── extracted_content/
```

In [ ]:
#@title 1. 安装系统依赖
!apt-get update -qq
!apt-get install -y -qq \
    python3-pip python3-venv git unzip \
    openjdk-17-jdk autoconf automake libtool \
    pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev \
    libtinfo5 cmake libffi-dev libssl-dev \
    libsqlite3-dev libjpeg-dev libpng-dev

!pip install --upgrade pip -q
!pip install buildozer cython -q

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

print('✅ 依赖安装完成')

In [ ]:
#@title 2. 上传项目文件
from google.colab import files
import zipfile, os, shutil

WORK_DIR = '/content/novel-generator'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

print('📤 请选择 novel_generator.zip 文件上传...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
zip_path = '/content/' + zip_name

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(WORK_DIR)

# 检查是否有多层目录
subdirs = [d for d in os.listdir(WORK_DIR) if os.path.isdir(os.path.join(WORK_DIR, d))]
py_files_in_root = [f for f in os.listdir(WORK_DIR) if f.endswith('.py')]

if not py_files_in_root and len(subdirs) == 1:
    inner = os.path.join(WORK_DIR, subdirs[0])
    for item in os.listdir(inner):
        src = os.path.join(inner, item)
        dst = os.path.join(WORK_DIR, item)
        shutil.move(src, dst)
    os.rmdir(inner)

os.chdir(WORK_DIR)
print(f'✅ 项目文件已解压到: {WORK_DIR}')
print('文件列表:')
for f in sorted(os.listdir(WORK_DIR)):
    print(f'  {f}')

In [ ]:
#@title 3. 验证必要文件
import os

required_files = [
    'main.py',
    'novel_core.py',
    'buildozer.spec',
]

optional_files = [
    '动态AI痕迹消除系统.py',
    'ultimate_ai_humanizer.py',
    '章节生成提示词辅助模块.py',
    '顶流作者人格建模辅助模块.py',
    '工具人角色检测系统.py',
]

print('=== 必要文件检查 ===')
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌'} {f}")
    if not exists:
        all_ok = False

print('\n=== 辅助模块检查 ===')
for f in optional_files:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '⚠️'} {f}")

if not all_ok:
    print('\n❌ 缺少必要文件，请检查 zip 包内容！')
else:
    print('\n✅ 所有必要文件就绪，可以开始构建！')

In [ ]:
#@title 4. 构建 APK（首次约15-30分钟，请耐心等待）
import os
os.chdir('/content/novel-generator')

!buildozer android debug 2>&1 | tail -100

In [ ]:
#@title 5. 查看构建结果
import os, glob

apk_files = glob.glob('/content/novel-generator/bin/*.apk')

if apk_files:
    print('✅ APK 构建成功！')
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / (1024 * 1024)
        print(f'  📦 {os.path.basename(apk)} ({size_mb:.1f} MB)')
else:
    print('❌ 未找到 APK 文件，请检查上一步的构建日志')

In [ ]:
#@title 6. 下载 APK
from google.colab import files
import glob, os

apk_files = glob.glob('/content/novel-generator/bin/*.apk')

if apk_files:
    for apk in apk_files:
        print(f'📥 下载: {os.path.basename(apk)}')
        files.download(apk)
else:
    print('❌ 未找到 APK 文件')